# Ordenar con `sorted`

En la práctica se ordena con `sorted` (devuelve una lista nueva) o `list.sort` (in situ). Ambos usan Timsort: un merge sort adaptativo que aprovecha los tramos ya ordenados, cuesta $O(n \log n)$ en el peor caso, $O(n)$ con datos ya ordenados, y es estable. Se personalizan con `key` (una función que da la clave de cada elemento) y `reverse`.

## 1. `key` y `reverse`

Ordenar palabras por longitud, sin distinguir mayúsculas, de mayor a menor y por la última letra.

In [1]:
palabras = ["Plátano", "kiwi", "manzana", "Uva", "fresa", "Aguacate", "pera"]
print(sorted(palabras))
print(sorted(palabras, key=str.lower))
print(sorted(palabras, key=len))
print(sorted(palabras, key=len, reverse=True))
print(sorted(palabras, key=lambda p: p[-1]))
assert sorted(palabras, key=len)[0] == "Uva"

['Aguacate', 'Plátano', 'Uva', 'fresa', 'kiwi', 'manzana', 'pera']
['Aguacate', 'fresa', 'kiwi', 'manzana', 'pera', 'Plátano', 'Uva']
['Uva', 'kiwi', 'pera', 'fresa', 'Plátano', 'manzana', 'Aguacate']
['Aguacate', 'Plátano', 'manzana', 'fresa', 'kiwi', 'pera', 'Uva']
['manzana', 'Uva', 'fresa', 'pera', 'Aguacate', 'kiwi', 'Plátano']


## 2. Varios criterios

Ordenar una lista de alumnos (nombre, curso, nota) por curso ascendente y, dentro del curso, por nota descendente y luego por nombre. Hacerlo con una tupla como clave y también con dos ordenaciones sucesivas aprovechando la estabilidad.

In [2]:
alumnos = [("Luis", 2, 7.5), ("Ana", 1, 9.0), ("Eva", 2, 9.0), ("Pau", 1, 6.0), ("Bea", 2, 7.5), ("Iker", 1, 9.0)]

por_tupla = sorted(alumnos, key=lambda a: (a[1], -a[2], a[0]))
for a in por_tupla:
    print(a)

# estabilidad: se ordena primero por el criterio menos importante
por_pasos = sorted(alumnos, key=lambda a: a[0])
por_pasos = sorted(por_pasos, key=lambda a: a[2], reverse=True)
por_pasos = sorted(por_pasos, key=lambda a: a[1])
assert por_pasos == por_tupla

('Ana', 1, 9.0)
('Iker', 1, 9.0)
('Pau', 1, 6.0)
('Eva', 2, 9.0)
('Bea', 2, 7.5)
('Luis', 2, 7.5)


## 3. `itemgetter` y `attrgetter`

Repetir ordenaciones con `operator.itemgetter` sobre diccionarios y `operator.attrgetter` sobre objetos de una `dataclass`, en vez de `lambda`.

In [3]:
from dataclasses import dataclass
from operator import attrgetter, itemgetter

ciudades = [{"nombre": "Las Palmas", "isla": "Gran Canaria", "habitantes": 378_000},
            {"nombre": "Santa Cruz", "isla": "Tenerife", "habitantes": 209_000},
            {"nombre": "La Laguna", "isla": "Tenerife", "habitantes": 158_000},
            {"nombre": "Telde", "isla": "Gran Canaria", "habitantes": 102_000}]
for c in sorted(ciudades, key=itemgetter("isla", "habitantes")):
    print(c["isla"], c["nombre"], c["habitantes"])

@dataclass
class Producto:
    nombre: str
    precio: float
    stock: int

productos = [Producto("teclado", 25.0, 4), Producto("ratón", 12.5, 10), Producto("monitor", 150.0, 2)]
print([p.nombre for p in sorted(productos, key=attrgetter("precio"))])
print(max(productos, key=attrgetter("stock")).nombre)

Gran Canaria Telde 102000
Gran Canaria Las Palmas 378000
Tenerife La Laguna 158000
Tenerife Santa Cruz 209000
['ratón', 'teclado', 'monitor']
ratón


## 4. `cmp_to_key`

Ordenar números para formar el mayor número posible al concatenarlos (por ejemplo `[3, 30, 34, 5, 9]` → `9534330`). No hay una clave sencilla, pero sí una comparación: `a` va antes que `b` si `ab > ba`. Usar `functools.cmp_to_key` y comprobarlo contra todas las permutaciones.

In [4]:
from functools import cmp_to_key
from itertools import permutations
import random

def compara(a, b):
    ab, ba = a + b, b + a
    return -1 if ab > ba else (1 if ab < ba else 0)

def mayor_numero(numeros):
    return "".join(sorted(map(str, numeros), key=cmp_to_key(compara))).lstrip("0") or "0"

print(mayor_numero([3, 30, 34, 5, 9]))
random.seed(1)
for _ in range(200):
    nums = [random.randint(0, 120) for _ in range(random.randint(1, 6))]
    mejor = max(int("".join(map(str, p))) for p in permutations(nums))
    assert int(mayor_numero(nums)) == mejor

9534330


## 5. Timsort aprovecha el orden

Medir `sorted` con un millón de números aleatorios, ya ordenados, en orden inverso y casi ordenados (pocos cambios) para ver que Timsort es mucho más rápido cuando hay tramos ordenados.

In [5]:
import time

n = 1_000_000
random.seed(2)
aleatoria = [random.random() for _ in range(n)]
ordenada = sorted(aleatoria)
casi = ordenada[:]
for _ in range(100):
    i, j = random.randrange(n), random.randrange(n)
    casi[i], casi[j] = casi[j], casi[i]

for nombre, datos in [("aleatoria", aleatoria), ("ordenada", ordenada), ("inversa", ordenada[::-1]), ("casi ordenada", casi)]:
    inicio = time.perf_counter()
    sorted(datos)
    print(f"{nombre:<15}{(time.perf_counter() - inicio) * 1000:8.1f} ms")

aleatoria         142.6 ms
ordenada           27.6 ms
inversa            30.9 ms
casi ordenada      32.5 ms


## 6. `sort` frente a `sorted` y errores típicos

Comprobar que `list.sort` devuelve `None`, que no se pueden ordenar mezclas de tipos que no se comparan y cómo ordenar con `None` usando una clave.

In [6]:
numeros = [3, 1, 2]
resultado = numeros.sort()
print(resultado, numeros)

try:
    sorted([3, "dos", 1])
except TypeError as e:
    print("TypeError:", e)

valores = [5, None, 2, None, 8]
print(sorted(valores, key=lambda x: (x is None, x if x is not None else 0)))

None [1, 2, 3]
TypeError: '<' not supported between instances of 'str' and 'int'
[2, 5, 8, None, None]
